In [1]:
import sys
print(sys.executable)

d:\Insura\.venv\Scripts\python.exe


## Imports & configuration

In [2]:
from faker import Faker
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import timedelta

fake = Faker("en_IN")
np.random.seed(42)
Faker.seed(42)

In [14]:
# -----------------------------
# Dataset sizes
# -----------------------------
N_LOCATIONS = 20
N_CUSTOMERS = 500
N_AGENTS = 30
N_VEHICLES = 500
N_POLICIES = 750
N_GARAGES = 30
N_HOSPITALS = 20
N_CLAIMS = 300

# -----------------------------
# Date range
# -----------------------------
START_DATE = pd.Timestamp("2024-01-01")
END_DATE = pd.Timestamp("2025-12-31")

# -----------------------------
# Output directory
# -----------------------------
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("InsuraFlow raw-data generation initialized.")
print(f"Output directory: {RAW_DIR.resolve()}")

InsuraFlow raw-data generation initialized.
Output directory: D:\Insura\data\raw


#### Locations

In [15]:
locations_master = [
    ("Mumbai", "Maharashtra", "MH", "400001", "West"),
    ("Pune", "Maharashtra", "MH", "411001", "West"),
    ("Nagpur", "Maharashtra", "MH", "440001", "West"),
    ("Nashik", "Maharashtra", "MH", "422001", "West"),
    ("Ahmedabad", "Gujarat", "GJ", "380001", "West"),
    ("Surat", "Gujarat", "GJ", "395001", "West"),
    ("Bengaluru", "Karnataka", "KA", "560001", "South"),
    ("Mysuru", "Karnataka", "KA", "570001", "South"),
    ("Chennai", "Tamil Nadu", "TN", "600001", "South"),
    ("Coimbatore", "Tamil Nadu", "TN", "641001", "South"),
    ("Hyderabad", "Telangana", "TS", "500001", "South"),
    ("Kochi", "Kerala", "KL", "682001", "South"),
    ("Delhi", "Delhi", "DL", "110001", "North"),
    ("Jaipur", "Rajasthan", "RJ", "302001", "North"),
    ("Lucknow", "Uttar Pradesh", "UP", "226001", "North"),
    ("Chandigarh", "Chandigarh", "CH", "160001", "North"),
    ("Kolkata", "West Bengal", "WB", "700001", "East"),
    ("Bhubaneswar", "Odisha", "OD", "751001", "East"),
    ("Bhopal", "Madhya Pradesh", "MP", "462001", "Central"),
    ("Indore", "Madhya Pradesh", "MP", "452001", "Central")
]

locations = pd.DataFrame(
    locations_master,
    columns=[
        "city",
        "state",
        "state_code",
        "pincode",
        "region"
    ]
)

locations.insert(
    0,
    "location_id",
    range(1, len(locations) + 1)
)

print(f"Locations generated: {len(locations)}")

display(locations.head())

Locations generated: 20


,location_id,city,state,state_code,pincode,region
0,1,Mumbai,Maharashtra,MH,400001,West
1,2,Pune,Maharashtra,MH,411001,West
2,3,Nagpur,Maharashtra,MH,440001,West
3,4,Nashik,Maharashtra,MH,422001,West
4,5,Ahmedabad,Gujarat,GJ,380001,West


### Generate master data

#### Customers + Agents

In [16]:
# ============================================================
# CUSTOMERS
# ============================================================

customer_first_names = [
    "Aarav", "Vivaan", "Aditya", "Arjun", "Rahul",
    "Rohan", "Vikram", "Karan", "Akash", "Nikhil",
    "Ananya", "Priya", "Sneha", "Neha", "Kavya", "Vedant",
    "Pooja", "Riya", "Isha", "Meera", "Aditi", "Meena", "Shubham"
]

customer_last_names = [
    "Sharma", "Patel", "Deshmukh", "Joshi", "Kulkarni",
    "Mehta", "Shah", "Verma", "Gupta", "Singh",
    "Iyer", "Nair", "Reddy", "Rao", "Kapoor", "Surve", "Patil", "Pawar"
]

customers = []

for i in range(1, N_CUSTOMERS + 1):

    first = np.random.choice(customer_first_names)
    last = np.random.choice(customer_last_names)

    location = locations.sample(
        n=1,
        random_state=1000 + i
    ).iloc[0]

    # Registration date
    registration_date = (
        START_DATE
        + pd.Timedelta(
            days=np.random.randint(
                0,
                (END_DATE - START_DATE).days + 1
            )
        )
    )

    # Customer must be at least 18 years old
    latest_dob = registration_date - pd.DateOffset(years=18)

    dob_start = pd.Timestamp("1960-01-01")

    dob_days = (
        latest_dob - dob_start
    ).days

    date_of_birth = (
        dob_start
        + pd.Timedelta(
            days=np.random.randint(
                0,
                dob_days + 1
            )
        )
    )

    customers.append({
        "customer_code": f"CUST{i:06d}",
        "first_name": first,
        "last_name": last,
        "date_of_birth": date_of_birth,
        "gender": np.random.choice(
            ["Male", "Female", "Other"],
            p=[0.50, 0.48, 0.02]
        ),
        "email": (
            f"{first.lower()}."
            f"{last.lower()}"
            f"{i}@insuraflow.test"
        ),
        "phone": (
            "+91"
            + str(
                fake.random_number(
                    digits=10,
                    fix_len=True
                )
            )
        ),
        "occupation": np.random.choice([
            "Software Engineer",
            "Teacher",
            "Business Owner",
            "Doctor",
            "Accountant",
            "Consultant",
            "Government Employee",
            "Sales Executive",
            "Student",
            "Retired"
        ]),
        "marital_status": np.random.choice(
            ["Single", "Married", "Divorced", "Widowed"],
            p=[0.35, 0.55, 0.07, 0.03]
        ),
        "location_id": int(location["location_id"]),
        "customer_type": np.random.choice(
            ["Individual", "Corporate"],
            p=[0.90, 0.10]
        ),
        "kyc_status": np.random.choice(
            ["Pending", "Verified", "Rejected"],
            p=[0.08, 0.88, 0.04]
        ),
        "registration_date": registration_date
    })


customers = pd.DataFrame(customers)

customers.insert(
    0,
    "customer_id",
    range(1, len(customers) + 1)
)

# KYC verification date
customers["kyc_verified_date"] = customers.apply(
    lambda row: (
        row["registration_date"]
        + pd.Timedelta(
            days=np.random.randint(1, 15)
        )
        if row["kyc_status"] == "Verified"
        else pd.NaT
    ),
    axis=1
)


# ============================================================
# AGENTS
# ============================================================

agents = []

for i in range(1, N_AGENTS + 1):

    first = np.random.choice(customer_first_names)
    last = np.random.choice(customer_last_names)

    location = locations.sample(
        n=1,
        random_state=2000 + i
    ).iloc[0]

    joining_start = pd.Timestamp("2018-01-01")
    joining_end = pd.Timestamp("2024-01-01")

    joining_date = (
        joining_start
        + pd.Timedelta(
            days=np.random.randint(
                0,
                (joining_end - joining_start).days + 1
            )
        )
    )

    agents.append({
        "agent_code": f"AGT{i:04d}",
        "first_name": first,
        "last_name": last,
        "email": (
            f"{first.lower()}."
            f"{last.lower()}"
            f".agent{i}@insuraflow.test"
        ),
        "phone": (
            "+91"
            + str(
                fake.random_number(
                    digits=10,
                    fix_len=True
                )
            )
        ),
        "location_id": int(location["location_id"]),
        "branch_name": f"{location['city']} Branch",
        "joining_date": joining_date.date(),
        "agent_status": np.random.choice(
            ["Active", "Inactive", "Suspended"],
            p=[0.85, 0.10, 0.05]
        )
    })


agents = pd.DataFrame(agents)

agents.insert(
    0,
    "agent_id",
    range(1, len(agents) + 1)
)


print(f"Customers generated: {len(customers)}")
print(f"Agents generated: {len(agents)}")

display(customers.head())
display(agents.head())

Customers generated: 500
Agents generated: 30


,customer_id,customer_code,first_name,last_name,date_of_birth,gender,email,phone,occupation,marital_status,location_id,customer_type,kyc_status,registration_date,kyc_verified_date
0,1,CUST000001,Akash,Kulkarni,1980-10-16,Male,akash.kulkarni1@insuraflow.test,+911168114160,Student,Single,2,Individual,Verified,2024-06-09,2024-06-22
1,2,CUST000002,Vedant,Shah,2007-01-07,Female,vedant.shah2@insuraflow.test,+918334179373,Accountant,Single,16,Individual,Verified,2025-01-23,2025-01-30
2,3,CUST000003,Karan,Patel,1990-10-27,Male,karan.patel3@insuraflow.test,+919551219870,Teacher,Single,7,Individual,Verified,2025-03-08,2025-03-09
3,4,CUST000004,Neha,Sharma,2007-07-26,Male,neha.sharma4@insuraflow.test,+913619094295,Software Engineer,Married,10,Individual,Verified,2025-10-15,2025-10-28
4,5,CUST000005,Ananya,Singh,1978-01-22,Female,ananya.singh5@insuraflow.test,+916857897124,Business Owner,Single,16,Individual,Verified,2024-12-15,2024-12-26


,agent_id,agent_code,first_name,last_name,email,phone,location_id,branch_name,joining_date,agent_status
0,1,AGT0001,Pooja,Deshmukh,pooja.deshmukh.agent1@insuraflow.test,+911626785308,14,Jaipur Branch,2021-09-16,Active
1,2,AGT0002,Arjun,Patil,arjun.patil.agent2@insuraflow.test,+916013117609,1,Mumbai Branch,2019-04-09,Active
2,3,AGT0003,Pooja,Shah,pooja.shah.agent3@insuraflow.test,+916480214504,13,Delhi Branch,2019-04-19,Active
3,4,AGT0004,Vedant,Kulkarni,vedant.kulkarni.agent4@insuraflow.test,+916582393412,1,Mumbai Branch,2020-12-22,Active
4,5,AGT0005,Riya,Singh,riya.singh.agent5@insuraflow.test,+914361527572,13,Delhi Branch,2018-09-17,Active


### Generate policies and vehicles

In [17]:
# ============================================================
# VEHICLES
# ============================================================

vehicle_catalog = [
    ("Maruti", "Swift", "Hatchback", "Petrol", 1197),
    ("Hyundai", "Creta", "SUV", "Petrol", 1497),
    ("Tata", "Nexon", "SUV", "Petrol", 1199),
    ("Honda", "City", "Sedan", "Petrol", 1498),
    ("Mahindra", "XUV700", "SUV", "Diesel", 2198),
    ("Toyota", "Innova", "MPV", "Diesel", 2393),
    ("Tata", "Nexon EV", "SUV", "Electric", 0),
    ("Hyundai", "i20", "Hatchback", "Petrol", 1197)
]

vehicles = []

for i in range(1, N_VEHICLES + 1):

    customer_id = np.random.randint(
        1,
        N_CUSTOMERS + 1
    )

    customer_location_id = int(
        customers.loc[
            customers["customer_id"] == customer_id,
            "location_id"
        ].iloc[0]
    )

    make, model, vehicle_type, fuel_type, engine_cc = (
        vehicle_catalog[
            np.random.randint(
                len(vehicle_catalog)
            )
        ]
    )

    manufacture_year = np.random.randint(
        2017,
        2025
    )

    # Generate a valid state-based registration number
    state_code = locations.loc[
        locations["location_id"] == customer_location_id,
        "state_code"
    ].iloc[0]

    registration_number = (
        f"{state_code}"
        f"{np.random.randint(10, 99)}"
        f"{chr(np.random.randint(65, 91))}"
        f"{chr(np.random.randint(65, 91))}"
        f"{np.random.randint(1000, 9999)}"
    )

    registration_date = (
        pd.Timestamp(
            f"{manufacture_year}-01-01"
        )
        + pd.Timedelta(
            days=np.random.randint(
                0,
                365
            )
        )
    )

    vehicles.append({
        "vehicle_id": i,
        "customer_id": customer_id,
        "registration_number": registration_number,
        "make": make,
        "model": model,
        "variant": np.random.choice(
            ["Base", "Mid", "Top"]
        ),
        "manufacture_year": manufacture_year,
        "registration_date": registration_date,
        "fuel_type": fuel_type,
        "vehicle_type": vehicle_type,
        "engine_capacity_cc": (
            engine_cc
            if engine_cc > 0
            else None
        ),
        "vehicle_value": round(
            np.random.uniform(
                350000,
                3000000
            ),
            2
        )
    })


vehicles = pd.DataFrame(vehicles)


# ============================================================
# POLICIES
# ============================================================

policies = []

policy_start_limit = pd.Timestamp(
    "2025-12-01"
)

for i in range(1, N_POLICIES + 1):

    customer_id = np.random.randint(
        1,
        N_CUSTOMERS + 1
    )

    customer_vehicles = vehicles[
        vehicles["customer_id"] == customer_id
    ]

    if len(customer_vehicles) == 0:

        vehicle_id = np.random.randint(
            1,
            N_VEHICLES + 1
        )

    else:

        vehicle_id = int(
            customer_vehicles.sample(
                n=1
            )["vehicle_id"].iloc[0]
        )

    agent_id = np.random.randint(
        1,
        N_AGENTS + 1
    )

    # Policy start date
    start_date = (
        START_DATE
        + pd.Timedelta(
            days=np.random.randint(
                0,
                (policy_start_limit - START_DATE).days + 1
            )
        )
    )

    end_date = (
        start_date
        + pd.DateOffset(
            years=1
        )
    )

    issue_date = (
        start_date
        - pd.Timedelta(
            days=np.random.randint(
                1,
                15
            )
        )
    )

    renewal_date = (
        end_date
        + pd.Timedelta(
            days=1
        )
    )

    policy_type = np.random.choice(
        [
            "Comprehensive",
            "Third Party",
            "Own Damage"
        ],
        p=[0.60, 0.25, 0.15]
    )

    vehicle_value = float(
        vehicles.loc[
            vehicles["vehicle_id"] == vehicle_id,
            "vehicle_value"
        ].iloc[0]
    )

    premium = round(
        vehicle_value
        * np.random.uniform(
            0.015,
            0.035
        ),
        2
    )

    sum_insured = round(
        vehicle_value
        * np.random.uniform(
            0.80,
            1.00
        ),
        2
    )

    policies.append({
        "policy_id": i,
        "policy_number": f"POL{i:08d}",
        "customer_id": customer_id,
        "vehicle_id": vehicle_id,
        "agent_id": agent_id,
        "policy_type": policy_type,
        "issue_date": issue_date,
        "start_date": start_date,
        "end_date": end_date,
        "renewal_date": renewal_date,
        "premium_amount": premium,
        "sum_insured": sum_insured,
        "deductible_amount": round(
            np.random.uniform(
                1000,
                15000
            ),
            2
        ),
        "policy_status": np.random.choice(
            [
                "Active",
                "Expired",
                "Cancelled",
                "Pending",
                "Lapsed"
            ],
            p=[
                0.45,
                0.35,
                0.08,
                0.07,
                0.05
            ]
        )
    })


policies = pd.DataFrame(policies)

print(f"Vehicles generated: {len(vehicles)}")
print(f"Policies generated: {len(policies)}")

display(vehicles.head())
display(policies.head())

Vehicles generated: 500
Policies generated: 750


,vehicle_id,customer_id,registration_number,make,model,variant,manufacture_year,registration_date,fuel_type,vehicle_type,engine_capacity_cc,vehicle_value
0,1,245,UP79HA6014,Tata,Nexon,Base,2021,2021-04-14,Petrol,SUV,1199.0,1700670.29
1,2,319,OD66BO5393,Hyundai,Creta,Mid,2020,2020-01-21,Petrol,SUV,1497.0,1649051.44
2,3,472,CH95NE4865,Hyundai,Creta,Mid,2022,2022-12-31,Petrol,SUV,1497.0,534268.70
3,4,425,DL64CL6029,Tata,Nexon,Top,2023,2023-08-07,Petrol,SUV,1199.0,2438921.05
4,5,149,DL33SH4278,Hyundai,i20,Top,2024,2024-12-25,Petrol,Hatchback,1197.0,2250854.90


,policy_id,policy_number,customer_id,vehicle_id,agent_id,policy_type,issue_date,start_date,end_date,renewal_date,premium_amount,sum_insured,deductible_amount,policy_status
0,1,POL00000001,475,29,17,Comprehensive,2024-01-15,2024-01-16,2025-01-16,2025-01-17,55702.71,1351896.67,11721.32,Pending
1,2,POL00000002,143,285,3,Third Party,2025-06-05,2025-06-08,2026-06-08,2026-06-09,27007.25,935335.44,12031.54,Pending
2,3,POL00000003,133,355,26,Own Damage,2024-07-05,2024-07-19,2025-07-19,2025-07-20,31782.68,1921808.64,13703.78,Active
3,4,POL00000004,139,79,20,Comprehensive,2024-08-19,2024-08-23,2025-08-23,2025-08-24,42798.84,2375537.74,4958.23,Lapsed
4,5,POL00000005,76,297,20,Comprehensive,2024-10-28,2024-11-02,2025-11-02,2025-11-03,43099.08,2649361.52,12939.20,Active


### Generate Coverage, Premium Payments, Garages, Hospitals

In [18]:
# ============================================================
# POLICY COVERAGE
# ============================================================

coverage_types = [
    "Own Damage",
    "Third Party Liability",
    "Theft",
    "Fire",
    "Personal Accident"
]

policy_coverage = []

coverage_id = 1

for _, policy in policies.iterrows():

    number_of_coverages = np.random.randint(
        2,
        5
    )

    selected_coverages = np.random.choice(
        coverage_types,
        size=number_of_coverages,
        replace=False
    )

    for coverage in selected_coverages:

        policy_coverage.append({
            "policy_coverage_id": coverage_id,
            "policy_id": int(
                policy["policy_id"]
            ),
            "coverage_type": coverage,
            "coverage_limit": round(
                policy["sum_insured"]
                * np.random.uniform(
                    0.30,
                    1.00
                ),
                2
            ),
            "deductible_amount": round(
                np.random.uniform(
                    500,
                    10000
                ),
                2
            ),
            "premium_component": round(
                policy["premium_amount"]
                / number_of_coverages,
                2
            ),
            "coverage_status": np.random.choice(
                [
                    "Active",
                    "Inactive",
                    "Expired"
                ],
                p=[0.85, 0.05, 0.10]
            )
        })

        coverage_id += 1


policy_coverage = pd.DataFrame(
    policy_coverage
)


# ============================================================
# PREMIUM PAYMENTS
# ============================================================

premium_payments = []

payment_id = 1

for _, policy in policies.iterrows():

    number_of_payments = np.random.choice(
        [1, 2, 3],
        p=[0.55, 0.30, 0.15]
    )

    remaining = float(
        policy["premium_amount"]
    )

    for p in range(number_of_payments):

        if p == number_of_payments - 1:

            amount = remaining

        else:

            amount = round(
                remaining
                * np.random.uniform(
                    0.25,
                    0.60
                ),
                2
            )

        remaining -= amount

        payment_date = (
            policy["issue_date"]
            + pd.Timedelta(
                days=np.random.randint(
                    1,
                    90
                )
            )
        )

        premium_payments.append({
            "premium_payment_id": payment_id,
            "policy_id": int(
                policy["policy_id"]
            ),
            "payment_date": payment_date,
            "amount": round(
                amount,
                2
            ),
            "payment_method": np.random.choice([
                "Bank Transfer",
                "Credit Card",
                "Debit Card",
                "UPI",
                "Cheque"
            ]),
            "payment_status": np.random.choice(
                [
                    "Completed",
                    "Pending",
                    "Failed",
                    "Refunded"
                ],
                p=[
                    0.80,
                    0.10,
                    0.07,
                    0.03
                ]
            ),
            "transaction_reference": (
                f"PREM-TXN-{payment_id:08d}"
            )
        })

        payment_id += 1


premium_payments = pd.DataFrame(
    premium_payments
)


# ============================================================
# GARAGES
# ============================================================

garages = []

for i in range(1, N_GARAGES + 1):

    location = locations.sample(
        n=1,
        random_state=3000 + i
    ).iloc[0]

    garages.append({
        "garage_id": i,
        "garage_name": (
            f"{fake.last_name()} Motors"
        ),
        "location_id": int(
            location["location_id"]
        ),
        "network_type": np.random.choice(
            [
                "Network",
                "Non-Network"
            ],
            p=[0.75, 0.25]
        ),
        "approved_status": np.random.choice(
            [
                "Approved",
                "Pending",
                "Suspended"
            ],
            p=[0.85, 0.10, 0.05]
        ),
        "contact_phone": (
            "+91"
            + str(
                fake.random_number(
                    digits=10,
                    fix_len=True
                )
            )
        )
    })


garages = pd.DataFrame(
    garages
)


# ============================================================
# HOSPITALS
# ============================================================

hospitals = []

for i in range(1, N_HOSPITALS + 1):

    location = locations.sample(
        n=1,
        random_state=4000 + i
    ).iloc[0]

    hospitals.append({
        "hospital_id": i,
        "hospital_name": (
            f"{fake.last_name()} "
            f"Multispeciality Hospital"
        ),
        "location_id": int(
            location["location_id"]
        ),
        "hospital_type": np.random.choice([
            "General",
            "Multispeciality",
            "Trauma Center"
        ]),
        "network_status": np.random.choice(
            [
                "Network",
                "Non-Network"
            ],
            p=[0.75, 0.25]
        ),
        "contact_phone": (
            "+91"
            + str(
                fake.random_number(
                    digits=10,
                    fix_len=True
                )
            )
        )
    })


hospitals = pd.DataFrame(
    hospitals
)


print(
    f"Policy coverage rows: {len(policy_coverage)}"
)
print(
    f"Premium payment rows: {len(premium_payments)}"
)
print(
    f"Garages: {len(garages)}"
)
print(
    f"Hospitals: {len(hospitals)}"
)

Policy coverage rows: 2249
Premium payment rows: 1211
Garages: 30
Hospitals: 20


### Claims, Assessments, Documents, Claim Payments

In [19]:
# ============================================================
# CLAIMS
# ============================================================

claims = []

for i in range(1, N_CLAIMS + 1):

    policy = policies.sample(
        n=1
    ).iloc[0]

    policy_start = pd.Timestamp(
        policy["start_date"]
    )

    policy_end = pd.Timestamp(
        policy["end_date"]
    )

    policy_days = (
        policy_end - policy_start
    ).days

    # Incident occurs during policy period
    incident_date = (
        policy_start
        + pd.Timedelta(
            days=np.random.randint(
                0,
                policy_days + 1
            )
        )
    )

    # Claim may be reported a few days after incident
    claim_date = (
        incident_date
        + pd.Timedelta(
            days=np.random.randint(
                0,
                15
            )
        )
    )

    estimated_amount = round(
        np.random.uniform(
            10000,
            500000
        ),
        2
    )

    claim_status = np.random.choice(
        [
            "Submitted",
            "Under Review",
            "Approved",
            "Rejected",
            "Settled"
        ],
        p=[
            0.15,
            0.20,
            0.25,
            0.10,
            0.30
        ]
    )

    approved_amount = (
        round(
            estimated_amount
            * np.random.uniform(
                0.60,
                0.95
            ),
            2
        )
        if claim_status
        in ["Approved", "Settled"]
        else None
    )

    incident_location = locations.sample(
        n=1
    ).iloc[0]

    claims.append({
        "claim_id": i,
        "claim_number": f"CLM{i:08d}",
        "policy_id": int(
            policy["policy_id"]
        ),
        "vehicle_id": int(
            policy["vehicle_id"]
        ),
        "claim_date": claim_date,
        "incident_date": incident_date,
        "claim_type": np.random.choice([
            "Accident",
            "Theft",
            "Fire",
            "Natural Disaster",
            "Third Party Damage"
        ]),
        "incident_location_id": int(
            incident_location["location_id"]
        ),
        "incident_description": fake.sentence(
            nb_words=12
        ),
        "estimated_amount": estimated_amount,
        "approved_amount": approved_amount,
        "claim_status": claim_status
    })


claims = pd.DataFrame(
    claims
)


# ============================================================
# CLAIM ASSESSMENTS
# ============================================================

assessment_candidates = claims[
    claims["claim_status"].isin(
        [
            "Under Review",
            "Approved",
            "Rejected",
            "Settled"
        ]
    )
]

claim_assessments = []

for i, (_, claim) in enumerate(
    assessment_candidates.iterrows(),
    start=1
):

    assessment_date = (
        pd.Timestamp(
            claim["claim_date"]
        )
        + pd.Timedelta(
            days=np.random.randint(
                1,
                10
            )
        )
    )

    if claim["claim_status"] in [
        "Approved",
        "Settled"
    ]:
        assessment_status = "Approved"

    elif claim["claim_status"] == "Rejected":
        assessment_status = "Rejected"

    else:
        assessment_status = "Requires Review"

    claim_assessments.append({
        "assessment_id": i,
        "claim_id": int(
            claim["claim_id"]
        ),
        "assessor_name": fake.name(),
        "assessment_date": assessment_date,
        "estimated_damage": float(
            claim["estimated_amount"]
        ),
        "approved_amount": claim[
            "approved_amount"
        ],
        "assessment_status": assessment_status,
        "remarks": fake.sentence(
            nb_words=10
        )
    })


claim_assessments = pd.DataFrame(
    claim_assessments
)


# ============================================================
# CLAIM DOCUMENTS
# ============================================================

claim_documents = []

document_types = [
    "Driving License",
    "FIR",
    "Vehicle Photos",
    "Repair Estimate",
    "Medical Report"
]

document_id = 1

for _, claim in claims.iterrows():

    number_of_documents = np.random.randint(
        1,
        4
    )

    selected_docs = np.random.choice(
        document_types,
        size=number_of_documents,
        replace=False
    )

    for document in selected_docs:

        claim_documents.append({
            "claim_document_id": document_id,
            "claim_id": int(
                claim["claim_id"]
            ),
            "document_type": document,
            "document_status": np.random.choice(
                [
                    "Uploaded",
                    "Missing",
                    "Rejected"
                ],
                p=[
                    0.85,
                    0.10,
                    0.05
                ]
            ),
            "uploaded_date": claim[
                "claim_date"
            ],
            "verification_status": np.random.choice(
                [
                    "Pending",
                    "Verified",
                    "Rejected"
                ],
                p=[
                    0.15,
                    0.80,
                    0.05
                ]
            )
        })

        document_id += 1


claim_documents = pd.DataFrame(
    claim_documents
)


# ============================================================
# CLAIM PAYMENTS
# ============================================================

settled_claims = claims[
    (
        claims["claim_status"] == "Settled"
    )
    &
    (
        claims["approved_amount"].notna()
    )
]

claim_payments = []

payment_candidates = settled_claims.sample(
    frac=0.90,
    random_state=42
)

for i, (_, claim) in enumerate(
    payment_candidates.iterrows(),
    start=1
):

    payment_amount = round(
        float(
            claim["approved_amount"]
        )
        * np.random.uniform(
            0.80,
            1.00
        ),
        2
    )

    if claim["claim_type"] in [
        "Accident",
        "Fire"
    ]:
        garage_id = int(
            np.random.choice(
                garages["garage_id"]
            )
        )
    else:
        garage_id = None

    if np.random.random() < 0.25:
        hospital_id = int(
            np.random.choice(
                hospitals["hospital_id"]
            )
        )
    else:
        hospital_id = None

    payment_date = (
        pd.Timestamp(
            claim["claim_date"]
        )
        + pd.Timedelta(
            days=np.random.randint(
                5,
                45
            )
        )
    )

    claim_payments.append({
        "claim_payment_id": i,
        "claim_id": int(
            claim["claim_id"]
        ),
        "garage_id": garage_id,
        "hospital_id": hospital_id,
        "payment_date": payment_date,
        "payment_amount": payment_amount,
        "payment_method": np.random.choice([
            "Bank Transfer",
            "Cheque",
            "UPI",
            "NEFT",
            "RTGS"
        ]),
        "payment_status": np.random.choice(
            [
                "Completed",
                "Processing",
                "Pending",
                "Failed"
            ],
            p=[
                0.80,
                0.08,
                0.07,
                0.05
            ]
        ),
        "transaction_reference": (
            f"CLM-TXN-{i:08d}"
        )
    })


claim_payments = pd.DataFrame(
    claim_payments
)


print(f"Claims: {len(claims)}")
print(
    f"Assessments: {len(claim_assessments)}"
)
print(
    f"Documents: {len(claim_documents)}"
)
print(
    f"Claim payments: {len(claim_payments)}"
)

Claims: 300
Assessments: 257
Documents: 601
Claim payments: 68


#### Save raw data 

In [20]:
datasets = {
    "locations": locations,
    "customers": customers,
    "agents": agents,
    "vehicles": vehicles,
    "policies": policies,
    "policy_coverage": policy_coverage,
    "premium_payments": premium_payments,
    "claims": claims,
    "claim_assessments": claim_assessments,
    "claim_documents": claim_documents,
    "garages": garages,
    "hospitals": hospitals,
    "claim_payments": claim_payments
}

print("Saving raw datasets...\n")

for name, df in datasets.items():

    csv_path = RAW_DIR / f"{name}.csv"
    parquet_path = RAW_DIR / f"{name}.parquet"

    df.to_csv(
        csv_path,
        index=False
    )

    df.to_parquet(
        parquet_path,
        index=False
    )

    print(
        f"{name:20} | "
        f"Rows: {len(df):5} | "
        f"Columns: {len(df.columns):2} | "
        f"CSV + Parquet saved"
    )

print("\nRAW DATA GENERATION COMPLETE.")
print(
    f"Location: {RAW_DIR.resolve()}"
)

Saving raw datasets...

locations            | Rows:    20 | Columns:  6 | CSV + Parquet saved
customers            | Rows:   500 | Columns: 15 | CSV + Parquet saved
agents               | Rows:    30 | Columns: 10 | CSV + Parquet saved
vehicles             | Rows:   500 | Columns: 12 | CSV + Parquet saved
policies             | Rows:   750 | Columns: 14 | CSV + Parquet saved
policy_coverage      | Rows:  2249 | Columns:  7 | CSV + Parquet saved
premium_payments     | Rows:  1211 | Columns:  7 | CSV + Parquet saved
claims               | Rows:   300 | Columns: 12 | CSV + Parquet saved
claim_assessments    | Rows:   257 | Columns:  8 | CSV + Parquet saved
claim_documents      | Rows:   601 | Columns:  6 | CSV + Parquet saved
garages              | Rows:    30 | Columns:  6 | CSV + Parquet saved
hospitals            | Rows:    20 | Columns:  6 | CSV + Parquet saved
claim_payments       | Rows:    68 | Columns:  9 | CSV + Parquet saved

RAW DATA GENERATION COMPLETE.
Location: D:\Insura\da

In [20]:
#   "customers": customers,
#     "contracts": contracts,
#     "vehicles": vehicles,
#     "claims": claims,
#     "payments": payments,
#     "agents": agents,
#     "hospitals": hospitals


contracts.head(20)

,contract_id,contract_code,customer_id,agent_id,policy_type,premium_amount,coverage_amount,start_date,end_date,status
0,1,POL00000001,52,5,Third Party,11150.77,949291.30,2024-08-27,2025-08-27,Active
1,2,POL00000002,93,10,Third Party,14717.73,2400705.23,2024-05-23,2025-05-23,Active
2,3,POL00000003,15,5,Own Damage,5189.76,542490.21,2024-04-06,2025-04-06,Cancelled
3,4,POL00000004,72,4,Own Damage,13168.72,316808.71,2024-07-19,2025-07-19,Active
4,5,POL00000005,61,2,Third Party,38121.63,1950366.22,2024-05-03,2025-05-03,Expired
5,6,POL00000006,21,20,Comprehensive,47427.13,263799.29,2024-07-05,2025-07-05,Active
6,7,POL00000007,83,10,Third Party,60600.26,2108989.58,2024-11-21,2025-11-21,Expired
7,8,POL00000008,87,19,Comprehensive,12468.95,1149264.15,2024-12-14,2025-12-14,Active
8,9,POL00000009,75,1,Third Party,64550.92,2297232.83,2024-09-15,2025-09-15,Expired
9,10,POL00000010,75,5,Third Party,57218.22,1831678.87,2024-05-27,2025-05-27,Expired
